In [ ]:
!pip install fastf1 scikit-learn pandas joblib pyarrow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.6/427.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.9 MB/s eta 0:00:00


In [16]:
"""
Train a PRE-RACE finishing-position predictor for F1 races.

Key improvements over v1:
- Recent races are weighted exponentially higher (2026 counts ~10x more than 2019)
- Added "last 3 races" rolling windows to catch hot/cold streaks faster
- Added current-season-only averages for driver and team
- Grid position gets extra emphasis since it's the strongest single predictor

RUN THIS IN GOOGLE COLAB (needs internet for FastF1 data download).
Outputs: f1_race_predictor.pkl + historical_features.parquet
Copy both into your GitHub repo root next to app.py.
"""

import os
import fastf1
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")
fastf1.set_log_level("WARNING")  # reduces noise in output

YEARS = range(2024, 2027)

# More features than v1 — shorter windows + current-season stats
FEATURES = [
    "GridPosition",
    # Recent form — 3-race window catches hot/cold streaks fast
    "DriverAvgFinishLast3",
    "DriverAvgGridLast3",
    # Medium-term form — 5-race window
    "DriverAvgFinishLast5",
    "DriverAvgGridLast5",
    # Current season only — most relevant to current car/regs
    "DriverSeasonAvgFinish",
    "DriverSeasonAvgGrid",
    # Reliability
    "DriverDNFRateLast10",
    # Season momentum
    "DriverPointsCumSeason",
    # Team form
    "TeamAvgFinishLast3",
    "TeamAvgFinishLast5",
    "TeamPointsCumSeason",
    "TeamSeasonAvgFinish",
    # Circuit history
    "CircuitAvgFinish",
    # Experience
    "DriverRaceCount",
]

# ── Recency weight function ───────────────────────────────────────────────────
# Each race gets a weight based on how recent it is.
# decay=0.25 means a race from 4 years ago has weight ~0.1x compared to today.
# Adjust decay lower (e.g. 0.15) to care more about history,
# or higher (e.g. 0.4) to care almost exclusively about recent races.
DECAY = 0.25

def compute_recency_weight(year, round_num, max_year, max_round, decay=DECAY):
    """Exponential recency weight. More recent = higher weight."""
    # Convert (year, round) to a single "race index" number
    race_idx     = year * 30 + round_num
    max_race_idx = max_year * 30 + max_round
    distance     = max_race_idx - race_idx   # 0 for most recent race
    return np.exp(-decay * distance / 30)    # normalised per ~season


def collect_race_results(years):
    import time
    rows = []
    for year in years:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        schedule = schedule[schedule["EventFormat"] != "testing"]
        for _, event in schedule.iterrows():
            rnd = int(event["RoundNumber"])
            if rnd == 0:
                continue
            try:
                session = fastf1.get_session(year, rnd, "R")
                session.load(laps=False, telemetry=False, weather=False, messages=False)
                results = session.results.copy()
                if results is None or results.empty:
                    continue
                results["Year"]      = year
                results["Round"]     = rnd
                results["EventName"] = event["EventName"]
                rows.append(results)
                print(f"  loaded {year} R{rnd} {event['EventName']}")
                time.sleep(8)   # ← stay under 500 calls/hour
            except Exception as e:
                print(f"  skipping {year} R{rnd}: {e}")
                time.sleep(15)     # ← back off more on errors
    return pd.concat(rows, ignore_index=True)


def build_features(all_results):
    df = all_results.copy()
    df["Position"]    = pd.to_numeric(df["Position"],    errors="coerce")
    df["GridPosition"]= pd.to_numeric(df["GridPosition"],errors="coerce")
    df["Points"]      = pd.to_numeric(df["Points"],      errors="coerce").fillna(0)
    df["DNF"]         = df["Status"].apply(
        lambda s: 0 if ("finish" in str(s).lower() or "lap" in str(s).lower()) else 1
    )
    df = df.sort_values(["Year", "Round"]).reset_index(drop=True)

    g_drv  = df.groupby("Abbreviation")
    g_team = df.groupby("TeamName")
    g_seas_drv  = df.groupby(["Year", "Abbreviation"])
    g_seas_team = df.groupby(["Year", "TeamName"])

    # ── Driver rolling features ───────────────────────────────────────────────
    df["DriverAvgFinishLast3"] = (
        g_drv["Position"].transform(lambda s: s.shift().rolling(3, min_periods=1).mean())
    )
    df["DriverAvgFinishLast5"] = (
        g_drv["Position"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
    )
    df["DriverAvgGridLast3"] = (
        g_drv["GridPosition"].transform(lambda s: s.shift().rolling(3, min_periods=1).mean())
    )
    df["DriverAvgGridLast5"] = (
        g_drv["GridPosition"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
    )
    df["DriverDNFRateLast10"] = (
        g_drv["DNF"].transform(lambda s: s.shift().rolling(10, min_periods=1).mean())
    )
    df["DriverPointsCumSeason"] = (
        g_seas_drv["Points"].transform(lambda s: s.shift().cumsum().fillna(0))
    )
    df["DriverRaceCount"] = g_drv.cumcount()

    # Current-season average finish (leakage-free: only uses races before this one)
    df["DriverSeasonAvgFinish"] = (
        g_seas_drv["Position"].transform(lambda s: s.shift().expanding().mean())
    )
    df["DriverSeasonAvgGrid"] = (
        g_seas_drv["GridPosition"].transform(lambda s: s.shift().expanding().mean())
    )

    # ── Team rolling features ─────────────────────────────────────────────────
    df["TeamAvgFinishLast3"] = (
        g_team["Position"].transform(lambda s: s.shift().rolling(3, min_periods=1).mean())
    )
    df["TeamAvgFinishLast5"] = (
        g_team["Position"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
    )
    df["TeamPointsCumSeason"] = (
        g_seas_team["Points"].transform(lambda s: s.shift().cumsum().fillna(0))
    )
    df["TeamSeasonAvgFinish"] = (
        g_seas_team["Position"].transform(lambda s: s.shift().expanding().mean())
    )

    # ── Circuit history ───────────────────────────────────────────────────────
    df["CircuitAvgFinish"] = (
        df.groupby(["Abbreviation", "EventName"])["Position"]
        .transform(lambda s: s.shift().expanding().mean())
    )

    return df


def main():
    print("Collecting race results (2019–2026)…")
    raw = collect_race_results(YEARS)

    print("Engineering features…")
    df = build_features(raw)

    model_df = df.dropna(subset=["Position"]).copy()
    for c in FEATURES:
        model_df[c] = model_df[c].fillna(model_df[c].median())

    X = model_df[FEATURES]
    y = model_df["Position"]

    # ── Compute sample weights ────────────────────────────────────────────────
    max_year  = int(model_df["Year"].max())
    max_round = int(model_df[model_df["Year"] == max_year]["Round"].max())

    weights = model_df.apply(
        lambda r: compute_recency_weight(int(r["Year"]), int(r["Round"]),
                                         max_year, max_round),
        axis=1
    ).values

    print(f"\nSample weight range: {weights.min():.3f} → {weights.max():.3f}")
    print(f"(Most recent race weight is {weights.max()/weights.min():.1f}x the oldest)\n")

    X_train, X_test, y_train, y_test, w_train, _ = train_test_split(
        X, y, weights, test_size=0.15, random_state=42
    )

    model = GradientBoostingRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.04,
        subsample=0.8,
        random_state=42
    )
    model.fit(X_train, y_train, sample_weight=w_train)

    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    print(f"Validation MAE (finishing position): {mae:.2f} positions")

    # Feature importance — useful to see what the model actually relies on
    importance = pd.Series(model.feature_importances_, index=FEATURES)
    print("\nTop features by importance:")
    print(importance.sort_values(ascending=False).to_string())

    joblib.dump({"model": model, "features": FEATURES}, "f1_race_predictor.pkl")
    print("\nSaved f1_race_predictor.pkl")

    df.to_parquet("historical_features.parquet", index=False)
    print("Saved historical_features.parquet")

    print("\nDone! Upload both files to your GitHub repo.")


if __name__ == "__main__":
    main()

INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

  loaded 2024 R1 Bahrain Grand Prix


INFO:fastf1.fastf1.core:Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  loaded 2024 R2 Saudi Arabian Grand Prix


INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  

...

  loaded 2026 R6 Monaco Grand Prix


INFO:fastf1.fastf1.core:Loading data for Barcelona Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

  loaded 2026 R7 Barcelona Grand Prix


INFO:fastf1.fastf1.core:Loading data for Austrian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fi

Engineering features…

Sample weight range: 0.577 → 1.000
(Most recent race weight is 1.7x the oldest)

Validation MAE (finishing position): 3.42 positions

Top features by importance:
GridPosition             0.422941
TeamSeasonAvgFinish      0.067116
DriverSeasonAvgGrid      0.059809
DriverRaceCount          0.052503
TeamAvgFinishLast5       0.048291
DriverSeasonAvgFinish    0.046301
DriverAvgGridLast5       0.046048
DriverAvgFinishLast5     0.043893
TeamPointsCumSeason      0.042907
DriverAvgGridLast3       0.034560
CircuitAvgFinish         0.031758
DriverPointsCumSeason    0.031470
TeamAvgFinishLast3       0.031300
DriverAvgFinishLast3     0.030829
DriverDNFRateLast10      0.010274

Saved f1_race_predictor.pkl
Saved historical_features.parquet

Done! Upload both files to your GitHub repo.
